In [2]:
import pandas as pd
from pathlib import Path

project_root = Path("..").resolve()
manifest_path = project_root / "data" / "processed" / "image_manifest.csv"

manifest = pd.read_csv(manifest_path, dtype={"camera_id": "string"})
timestamp = pd.to_datetime(manifest["timestamp"])

manifest["date"] = timestamp.dt.normalize()
manifest["hour"] = timestamp.dt.hour
manifest["is_weekend"] = timestamp.dt.dayofweek >= 5

coverage = (
    manifest.groupby(["camera_id", "hour", "is_weekend"], as_index=False)
    .agg(n_images=("date", "size"), n_days=("date", "nunique"))
)

coverage.head(10)

C:\Users\NEXT Speed\AppData\Local\Temp\ipykernel_1960\721157966.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  manifest["is_weekend"] = timestamp.dt.dayofweek >= 5


,camera_id,hour,is_weekend,n_images,n_days
0,1028,6,False,22,4
1,1028,6,True,12,2
2,1028,7,False,24,4
3,1028,7,True,11,2
4,1028,8,False,25,5
5,1028,8,True,12,2
6,1028,9,False,30,5
7,1028,9,True,12,2
8,1028,10,False,30,5
9,1028,10,True,12,2


In [3]:
days_by_type = coverage.pivot(
    index=["camera_id", "hour"],
    columns="is_weekend",
    values="n_days",
).reindex(columns=[False, True]).fillna(0)

# count camera weekday and weekend
has_both = days_by_type.gt(0).all(axis=1)
has_both.groupby(level="hour").sum().rename("n_cameras")

hour
6     100
7     100
8     100
9     100
10    100
11    100
12    100
13    100
14    100
15    100
16    100
17    100
18    100
19    100
20    100
21     70
Name: n_cameras, dtype: int64

In [4]:
print(has_both.groupby(level="hour").sum().rename("n_cameras"))

hour
6     100
7     100
8     100
9     100
10    100
11    100
12    100
13    100
14    100
15    100
16    100
17    100
18    100
19    100
20    100
21     70
Name: n_cameras, dtype: int64


In [5]:
import pandas as pd

# path
manifest = pd.read_csv(
    manifest_path,
    dtype={"camera_id": "string"},
    parse_dates=["timestamp"],
)

manifest["hour"] = manifest["timestamp"].dt.hour
manifest["is_weekend"] = manifest["timestamp"].dt.dayofweek >= 5

# Seed (same data)
camera_ids = (
    manifest["camera_id"]
    .drop_duplicates()
    .sort_values()
    .sample(n=5, random_state=42)
)

hours = [7, 12, 17, 20]
group_columns = ["camera_id", "hour", "is_weekend"]

candidates = manifest.loc[
    manifest["camera_id"].isin(camera_ids)
    & manifest["hour"].isin(hours)
]

pilot = (
    candidates.groupby(group_columns)
    .sample(n=1, random_state=42)
    .sort_values(group_columns)
    .reset_index(drop=True)
)

pilot = pilot[
    ["camera_id", "timestamp", "image_path", "hour", "is_weekend"]
].copy()

# Ground truth
pilot["manual_count"] = pd.NA

pilot.to_csv(project_root / "data/interim/vehicle_count_pilot.csv", index=False)

print(f"Selected {len(pilot)} images from {pilot['camera_id'].nunique()} cameras")
print(pilot.head())

Selected 40 images from 5 cameras
  camera_id           timestamp  \
0       186 2026-08-26 07:25:12   
1       186 2026-08-30 07:20:10   
2       186 2026-08-26 12:40:10   
3       186 2026-08-30 12:50:10   
4       186 2026-08-31 17:55:11   

                                          image_path  hour  is_weekend  \
0  data/raw/cctv_images/186/186_โครงสร้างป้ายบอกท...     7       False   
1  data/raw/cctv_images/186/186_โครงสร้างป้ายบอกท...     7        True   
2  data/raw/cctv_images/186/186_โครงสร้างป้ายบอกท...    12       False   
3  data/raw/cctv_images/186/186_โครงสร้างป้ายบอกท...    12        True   
4  data/raw/cctv_images/186/186_โครงสร้างป้ายบอกท...    17       False   

  manual_count  
0         <NA>  
1         <NA>  
2         <NA>  
3         <NA>  
4         <NA>  


C:\Users\NEXT Speed\AppData\Local\Temp\ipykernel_1960\3940733066.py:11: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  manifest["is_weekend"] = manifest["timestamp"].dt.dayofweek >= 5


In [6]:
# second set for train
pilot_path = project_root / "data/interim/vehicle_count_pilot.csv"
old_pilot = pd.read_csv(pilot_path, dtype={"camera_id": "string"})
old_camera_ids = old_pilot["camera_id"].dropna().unique()

hours = [7, 12, 17, 20]

candidates = manifest.loc[
    ~manifest["camera_id"].isin(old_camera_ids)
    & manifest["hour"].isin(hours)
].copy()

# select only camera all 4 hour 
hour_counts = candidates.groupby("camera_id")["hour"].nunique()
eligible_camera_ids = hour_counts.loc[hour_counts == len(hours)].index

if len(eligible_camera_ids) < 6:
    raise ValueError(
        f"need 6 cameras but found cameras already had 4 hours of use. "
        f"only {len(eligible_camera_ids)} cameras"
    )

new_camera_ids = (
    pd.Series(eligible_camera_ids)
    .sort_values()
    .sample(n=6, random_state=42)
)

# 4 img/camera
validation = (
    candidates.loc[candidates["camera_id"].isin(new_camera_ids)]
    .sort_values(["camera_id", "timestamp", "image_path"])
    .groupby(["camera_id", "hour"])
    .sample(n=1, random_state=42)
    .sort_values(["camera_id", "hour"])
    .reset_index(drop=True)
)

validation = validation[
    ["camera_id", "timestamp", "image_path", "hour", "is_weekend"]
].copy()

validation["manual_count"] = pd.NA
validation["notes"] = ""

output_path = project_root / "data/interim/vehicle_count_validation.csv"
validation.to_csv(output_path, index=False, encoding="utf-8-sig")

print(
    f"Selected {len(validation)} images "
    f"from {validation['camera_id'].nunique()} new cameras"
)
print(f"Saved to: {output_path}")
print(validation.head())

Selected 24 images from 6 new cameras
Saved to: D:\01University\dsde-project-v2\data\interim\vehicle_count_validation.csv
  camera_id           timestamp  \
0      1028 2026-08-30 07:30:58   
1      1028 2026-08-25 12:05:56   
2      1028 2026-08-28 17:05:56   
3      1028 2026-08-26 20:10:55   
4       127 2026-08-28 07:16:15   

                                          image_path  hour  is_weekend  \
0  data/raw/cctv_images/1028/1028_แยกสะพานแดง_แยก...     7        True   
1  data/raw/cctv_images/1028/1028_แยกสะพานแดง_แยก...    12       False   
2  data/raw/cctv_images/1028/1028_แยกสะพานแดง_แยก...    17       False   
3  data/raw/cctv_images/1028/1028_แยกสะพานแดง_แยก...    20       False   
4  data/raw/cctv_images/127/127_สะพานลอยซอยรัชดาภ...     7       False   

  manual_count notes  
0         <NA>        
1         <NA>        
2         <NA>        
3         <NA>        
4         <NA>        
